In [ ]:
!pip install -q groq sentence-transformers pymupdf gradio huggingface_hub gtts

In [ ]:
import os
from google.colab import userdata
from groq import Groq
from huggingface_hub import InferenceClient
from huggingface_hub import HfApi, InferenceClient

# Automatically fetch keys by variable name from Colab Secrets
try:
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    HF_TOKEN = userdata.get('HUGGINGFACE_TOKEN')
    print("✅ Secrets loaded successfully from Colab!")
except Exception as e:
    print("⚠️ Could not load secrets. Make sure 'Notebook access' is enabled in the Secrets tab.")

# Initialize API Clients
groq_client = Groq(api_key=GROQ_API_KEY)
hf_client = InferenceClient(token=HF_TOKEN)

# Verify Connections
try:
    groq_client.chat.completions.create(
        model="qwen/qwen3.6-27b",
        messages=[{"role": "user", "content": "ping"}],
        max_tokens=5
    )
    print("✅ Groq Connected!")
except Exception as e:
    print("❌ Groq Connection Issue:", e)


try:
  api = HfApi(token=HF_TOKEN)
  api.model_info("facebook/mms-tts-eng")
  print("✅ Hugging Face Connected!")

except Exception as e:
  print("Hugging Face Warning (fallback active):", e)





⚠️ Could not load secrets. Make sure 'Notebook access' is enabled in the Secrets tab.
✅ Groq Connected!
✅ Hugging Face Connected!


In [ ]:
import fitz  # PyMuPDF
import numpy as np
from sentence_transformers import SentenceTransformer

# Load Hugging Face Embedding Model
print("Loading MiniLM Embedding Model...")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# Vector Store Storage
current_chunks = []
current_embeddings = None

# Default Fallback Schedule
DEFAULT_FEST_SCHEDULE = """
COLLEGE EVENTS 2026 SCHEDULE:
1. CodeQuest (Hackathon): Starts at 10:00 AM on 25th August 2026 in Computer Lab 3, CS Block. First prize is ₹15,000.
2. RoboWars: Starts at 11:30 AM on 27th August 2026 in the Mechanical Workshop Arena. Safety goggles are mandatory.
3. Battle of the Bands: Starts at 3:00 PM on 25th August 2026 at the Main Open Amphitheatre.
4. Registration Desk: Located at the Main Entrance Foyer. Spot registrations close at 11:00 AM.
5. General Rules: College ID cards are strictly required. Food stalls open near the Sports Ground from 12:30 PM.
"""

def setup_knowledge_base(pdf_path=None):
    global current_chunks, current_embeddings
    text = ""

    if pdf_path and os.path.exists(pdf_path):
        doc = fitz.open(pdf_path)
        for page in doc:
            text += page.get_text() + "\n"
    else:
        text = DEFAULT_FEST_SCHEDULE

    words = text.split()
    current_chunks = [" ".join(words[i:i + 45]) for i in range(0, len(words), 45)]
    current_embeddings = embed_model.encode(current_chunks)
    return f"Indexed {len(current_chunks)} document chunks."

# Load default knowledge base
setup_knowledge_base()
print("✅ Knowledge Base Initialized!")

def retrieve_context(query, top_k=10):
    global current_chunks, current_embeddings
    query_vector = embed_model.encode([query])
    scores = np.dot(current_embeddings, query_vector.T).flatten()
    top_indices = np.argsort(scores)[::-1][:top_k]
    return "\n---\n".join([current_chunks[idx] for idx in top_indices])

Loading MiniLM Embedding Model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Knowledge Base Initialized!


In [ ]:
import os
import re
import gradio as gr
from gtts import gTTS

def speech_to_text(audio_path):
    """Whisper transcription via Groq."""
    if not audio_path:
        return ""
    try:
        with open(audio_path, "rb") as file_obj:
            transcription = groq_client.audio.transcriptions.create(
                file=(os.path.basename(audio_path), file_obj.read()),
                model="whisper-large-v3-turbo",
                response_format="text"
            )
        return str(transcription).strip()
    except Exception as e:
        print(f"STT Error: {e}")
        return ""

def text_to_speech(text, output_file="response.mp3"):
    """Text-to-Speech synthesis with automatic local fallback."""
    if not text or not text.strip():
        return None

    # Attempt 1: Hugging Face Voice API
    if hf_client:
        try:
            audio_bytes = hf_client.text_to_speech(
                text=text,
                model="facebook/mms-tts-eng"
            )
            with open(output_file, "wb") as f:
                f.write(audio_bytes)
            return output_file
        except Exception:
            pass  # Fail gracefully to gTTS

    # Attempt 2: Local gTTS Fallback
    try:
        tts = gTTS(text=text, lang="en", slow=False)
        tts.save(output_file)
        return output_file
    except Exception as e:
        print(f"TTS Error: {e}")
        return None

def generate_llm_response(query, context):
    """Groq LLM response generation with regex cleaning."""
    system_prompt = f"""You are the official Fest Guide.
Answer the user's question directly using ONLY the provided schedule details.
Do NOT output any thinking blocks, internal reasoning, or special XML tags.
Keep your response strictly to 1-2 concise sentences for voice delivery.

Event Details:
{context}"""

    try:
        completion = groq_client.chat.completions.create(
            model="qwen/qwen3.6-27b",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": query}
            ],
            temperature=0.1
        )
        raw_text = completion.choices[0].message.content

        # Remove <think>...</think> and any trailing whitespace
        clean_text = re.sub(r"<think>.*?</think>", "", raw_text, flags=re.DOTALL).strip()
        return clean_text if clean_text else "I could not find relevant details in the schedule."
    except Exception as e:
        return f"LLM Processing Error: {str(e)}"

def run_multimodal_agent(text_input, audio_input):
    # Step 1: Input Modality Detection
    user_query = ""
    if audio_input:
        user_query = speech_to_text(audio_input)
    elif text_input and text_input.strip():
        user_query = text_input.strip()

    if not user_query:
        return "No input detected", "Please record audio or type a query.", None

    # Step 2: Document Retrieval (RAG Tool)
    context = retrieve_context(user_query)

    # Step 3: LLM Reasoning Core
    answer = generate_llm_response(user_query, context)

    # Step 4: Voice Synthesis (TTS Modality)
    audio_path = text_to_speech(answer)

    return user_query, answer, audio_path

def handle_pdf_upload(file):
    if file is None:
        return "No file selected."
    file_path = file.name if hasattr(file, "name") else file
    return setup_knowledge_base(file_path)

# Build Gradio UI
with gr.Blocks(title="Voice-Guided Student Event Information Assistant") as demo:
    gr.Markdown("# 🎤 Voice-Guided Student Event Information Assistant")
    gr.Markdown("An end-to-end multimodal agent utilizing **Groq Whisper**, **RAG Embeddings**, and **Voice Synthesis**.")

    with gr.Row():
        with gr.Column(scale=1):
            pdf_input = gr.File(label="Upload Event Brochure (PDF)", file_types=[".pdf"], type="filepath")
            pdf_status = gr.Textbox(label="Brochure Status", value="Default 2026 Fest Schedule Active", interactive=False)
            pdf_input.upload(fn=handle_pdf_upload, inputs=[pdf_input], outputs=[pdf_status])

            gr.Markdown("---")
            text_box = gr.Textbox(label="Option A: Type Query", placeholder="e.g., Which cultural events are happening?")
            mic_input = gr.Audio(label="Option B: Speak Query", sources=["microphone"], type="filepath")
            submit_btn = gr.Button("Ask Assistant", variant="primary")

        with gr.Column(scale=1):
            detected_query = gr.Textbox(label="Detected Query (STT Output)", interactive=False)
            response_text = gr.Textbox(label="Assistant Response (Text)", interactive=False)
            response_audio = gr.Audio(label="Assistant Response (Voice)", autoplay=True)

    submit_btn.click(
        fn=run_multimodal_agent,
        inputs=[text_box, mic_input],
        outputs=[detected_query, response_text, response_audio]
    )

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://5c4fd67cdd59f6e549.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
